# Sampling Test

## Information

Técnicas de muestreo del módulo `scipy.stats.qmc`:
* Sobol: <--
    * Permite ampliar progresivamente el número de muestras
    * **Se recomienda que el número total de muestras sea una potencia de 2**.
* Latin Hypercube (LHS)
    * Funciona con muestras bajas
    * **NO** ampliar el número de muestras


<font color=fa5>**Nota**</font>:
Las siguientes funciones se han movido a AUX (el cuaderno sigue usando las funciones locales):
*  `aux.parameters` $\longleftarrow$ `flatten_params`, `restore_flat_params`, `add_truncnorm_config`, `add_beta_config`. 
*  `aux.sampling` $\longleftarrow$ `create_sobol`, `_uniform_sample`, `_truncnorm_sample`, `_beta_sample`, `_sample_from_pdf`, `sample_params`, `sample_weight`.

## Initialization

**Environment**

In [ ]:
from dotenv import dotenv_values, find_dotenv
from pathlib import Path
import os, sys

# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))

**Modules**

In [ ]:
import numpy as np
from scipy.stats import qmc
from scipy.stats import beta, truncnorm, uniform
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from ipywidgets import FloatSlider, interact, ToggleButton

from copy import deepcopy

In [ ]:
from aux.environment import resolve_directory

**Console**

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

**Set main directory paths**

In [ ]:

WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Basic Sobol Sampling

In [ ]:
N_VARIABLES = 30
M_SAMPLES = 2**10
SEED = 42

In [ ]:
def create_sobol(n_variables, seed=SEED):
    return qmc.Sobol(
        d=n_variables, scramble=True, seed=seed
    )


In [ ]:
sampler = create_sobol(N_VARIABLES)

m = int(np.log2(M_SAMPLES))-1
batch11 = sampler.random_base2(m=m)
batch12 = sampler.random_base2(m=m)
batch1 = np.vstack((batch11, batch12))    

pprint(f'Batch shape: {batch1.shape}, Batch size: {batch1.nbytes / 1024**2:.2f} MB')

In [ ]:
sampler = create_sobol(N_VARIABLES)

# Puedes trabajar sin la restricción de que sea una potencia de 2, **pero no es recomendable**.
batch2 = np.vstack([sampler.random(M_SAMPLES // 4) for _ in range(4)])

pprint(f'Batch shape: {batch2.shape}, Batch size: {batch2.nbytes / 1024**2:.2f} MB')

In [ ]:
# Test para verificar como aplicar el número de muestras en distintos ficheros.

batch3_parts = []
for i in range(4):
    sampler = create_sobol(N_VARIABLES)
    if i != 0: 
        sampler.fast_forward(i * (M_SAMPLES // 4))
    batch3_parts.append(sampler.random(M_SAMPLES // 4))

batch3 = np.vstack(batch3_parts)

In [ ]:
## Proyección PCA de los tres lotes
pca = PCA(n_components=2).fit(batch1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)

for ax, (name, batch) in zip(axes, [('Batch 1', batch1), ('Batch 2', batch2), ('Batch 3', batch3)]):
    projected = pca.transform(batch)
    ax.scatter(projected[:, 0], projected[:, 1], s=1)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])


fig.suptitle('Sobol Sampling PCA Projection')
fig.tight_layout()
plt.show()

## Distributions

In [ ]:
MIN = 0.0
MAX = 1.0
PEAK = 0.2*MAX
LAMBDA = 3 
STD = 0.3*(MAX-MIN)

In [ ]:
relative_peak = (PEAK - MIN) / (MAX - MIN)

beta_a = 1 + relative_peak * LAMBDA
beta_b = 1 + (1 - relative_peak) * LAMBDA

# Truncated normal distribution
trunc_a = (MIN - PEAK) / STD
trunc_b = (MAX - PEAK) / STD

distributions = [
    ("Asymmetric Beta", beta(beta_a, beta_b, loc=MIN, scale=MAX - MIN)),
    ("Truncated Normal", truncnorm(trunc_a, trunc_b, loc=PEAK, scale=STD)),
    ("Uniform", uniform(loc=MIN, scale=MAX - MIN)),
]

In [ ]:
x = np.linspace(MIN, MAX, 1000)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))

for ax, (title, dist) in zip(axes.flat, distributions):
    y = dist.pdf(x)
    ax.plot(x, y)
    ax.fill_between(x, y, alpha=0.2)
    ax.set_title(title)
    ax.set_xlim(MIN, MAX)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Parameter sampling

In [ ]:
PARAMS = {
    "V1": { "min": 10, "max": 20, "default": 15 },
    "V2": { "min": 0.1, "max": 1.0, "default": 0.5 },
}

PARAMS2 = {
    "V1": { "min": 10, "max": 20, "default": 15 },
    "V2": { "min": [0.2, 0.4], "max": [0.8, 1.0], "default": [0.5, 0.7] },
}

In [ ]:
def flatten_params(params):
    flat = {}
    for name, config in params.items():
        first = next(iter(config.values()))
        if isinstance(first, list):
            for i in range(len(first)):
                flat[f"{name}*{i:02}"] = {k: v[i] for k, v in config.items()}
        else:
            flat[name] = config
    return flat


def restore_flat_params(flat_params):
    result = {}
    for name, config in sorted(flat_params.items()):
        base, sep, _ = name.partition("*")
        if not sep:
            result[name] = config
        else:
            if isinstance(config, dict):
                group = result.setdefault(base, {})
                for k, v in config.items():
                    group.setdefault(k, []).append(v)
            else:
                result.setdefault(base, []).append(config)


    return result

flattened_params = flatten_params(PARAMS2)
pprint('Flattened PARAMS2:', flattened_params)
pprint('Restored sample:', restore_flat_params(flattened_params))


In [ ]:
def add_truncnorm_config(params, std_factor=0.2, names=None):
    result = deepcopy(params)

    selected_names = result if names is None else names

    for name in selected_names:
        config = result[name]
        config["std"] = (config["max"] - config["min"]) * std_factor
        config["method"] = "truncnorm"
        config["a"] = (config["min"] - config["default"]) / config["std"] 
        config["b"] = (config["max"] - config["default"]) / config["std"] 
        config["loc"] = config["default"]
        config["scale"] = config["std"]
        config["config_source"] = "simple"

    return result

def add_beta_config(params, lambda_, names=None):
    result = deepcopy(params)

    selected_names = result if names is None else names

    for name in selected_names:
        config = result[name]
        config["lambda"] = lambda_
        config["method"] = "beta"
        scale = config["max"] - config["min"]
        relative_peak = (config["default"] - config["min"]) / scale
        config["a"] = 1 + relative_peak * lambda_
        config["b"] = 1 + (1 - relative_peak) * lambda_
        config["loc"] = config["min"]
        config["scale"] = scale
        config["config_source"] = "simple"

    return result

def add_truncnorm_config_pdf(params, a, b, loc=0.0, scale=1.0, names=None):
    result = deepcopy(params)
    selected_names = result if names is None else names

    for name in selected_names:
        config = result[name]
        config.update(a=a, b=b, loc=loc, scale=scale, method="truncnorm",
                      config_source="raw_pdf")

    return result


def add_beta_config_pdf(params, a, b, loc=0.0, scale=1.0, names=None):
    """Use SciPy PDF parameters a, b, loc, scale; preserve min/max as clamps."""
    result = deepcopy(params)
    selected_names = result if names is None else names

    for name in selected_names:
        config = result[name]
        config.update(a=a, b=b, loc=loc, scale=scale, method="beta",
                      config_source="raw_pdf")

    return result


beta_params = add_beta_config(PARAMS, lambda_=5)
truncnorm_params = add_truncnorm_config(PARAMS, std_factor=0.2)

pprint('Beta parameters:', beta_params)
pprint('Truncated Normal parameters:', truncnorm_params)

# Equivalent PDF configurations for V1.
beta_pdf_params = add_beta_config_pdf(PARAMS, a=3.5, b=3.5, loc=10, scale=10, names=["V1"])
truncnorm_pdf_params = add_truncnorm_config_pdf(PARAMS, a=-2.5, b=2.5, loc=15, scale=2, names=["V1"])

pprint('Beta PDF parameters:', beta_pdf_params)
pprint('Truncated Normal PDF parameters:', truncnorm_pdf_params)


In [ ]:
# Configure scalar and list-valued parameters without modifying the input.
def add_beta_config_safe(params, lambda_, names=None):
    flat = flatten_params(params)
    return restore_flat_params(add_beta_config(flat, lambda_=lambda_, names=names))


def add_truncnorm_config_safe(params, std_factor=0.2, names=None):
    flat = flatten_params(params)
    return restore_flat_params(add_truncnorm_config(flat, std_factor=std_factor, names=names))

def add_beta_config_pdf_safe(params, a, b, loc=0.0, scale=1.0, names=None):
    flat = flatten_params(params)
    return restore_flat_params(add_beta_config_pdf(flat, a, b, loc=loc, scale=scale, names=names))


def add_truncnorm_config_pdf_safe(params, a, b, loc=0.0, scale=1.0, names=None):
    flat = flatten_params(params)
    return restore_flat_params(add_truncnorm_config_pdf(flat, a, b, loc=loc, scale=scale, names=names))


beta_params2 = add_beta_config_safe(PARAMS2, lambda_=5)
truncnorm_params2 = add_truncnorm_config_safe(PARAMS2, std_factor=0.2)

pprint('Beta parameters with lists:', beta_params2)
pprint('Truncated Normal parameters with lists:', truncnorm_params2)


### Dependent configurations

`affine('X', factor=2, offset=1)` uses `2 * sampled_X + 1` for a peak or bound.
References are stored in `afin_default`, `afin_min` and `afin_max`, separately
from the fixed `default`, `min` and `max` values. They are evaluated during each
sample without changing the configuration dictionary. `afin_default` sets the
beta mode or normal location before truncation. For a uniform distribution, it
preserves the interval width, or determines the opposite bound if one is supplied.

Ordinary parameters are sampled first. Dependent parameters are then sampled in
dictionary order, so references to other dependent parameters must follow that order.
Use flattened names such as `V2*00` when referencing list entries.


In [ ]:
def affine(parameter, factor=1.0, offset=0.0):
    return {"parameter": parameter, "factor": factor, "offset": offset}


def _add_dependent_config(params, method, afin_default, afin_min, afin_max, names, **settings):
    result = deepcopy(params)
    selected_names = result if names is None else names

    for name in selected_names:
        config = result[name]
        config.update(settings)
        config["method"] = method
        config["config_source"] = "affine"
        for key, reference in (("afin_default", afin_default), ("afin_min", afin_min), ("afin_max", afin_max)):
            if reference is not None:
                config[key] = reference

    return result

def add_beta_config_dependent(params, lambda_, afin_default=None, afin_min=None, afin_max=None, names=None):
    return _add_dependent_config(params, "beta", afin_default, afin_min, afin_max, names,
                                 **{"lambda": lambda_}) #Asi se evita el problema usar una palabra reservada.


def add_truncnorm_config_dependent(params, std_factor=0.2, afin_default=None, afin_min=None, afin_max=None, names=None):
    return _add_dependent_config(params, "truncnorm", afin_default, afin_min, afin_max, names,
                                 std_factor=std_factor)


def add_uniform_config_dependent(params, afin_default=None, afin_min=None, afin_max=None, names=None):
    return _add_dependent_config(params, "uniform", afin_default, afin_min, afin_max, names)





In [ ]:
#TODO: Crear una clase que ya tenga creadas las distribuciones por
#parametro.

def _uniform_sample(s, minimum, maximum):
    scale= maximum - minimum
    return minimum + s * scale, 1/scale


def _truncnorm_sample(s, a, b, loc, scale):
    # TODO
    # Tighten even hardcoded a/b to min/max, preserving any narrower original bounds.
    # a = max(a, (minimum - loc) / scale)
    # b = min(b, (maximum - loc) / scale)
    dist = truncnorm(a, b, loc=loc, scale=scale)
    value = dist.ppf(s)
    return value, dist.pdf(value)


def _beta_sample(s, a, b, loc, scale):
    # Truncate through the CDF and normalize the PDF, keeping a, b, loc and scale.
    # dist = beta(a, b, loc=loc, scale=scale)
    # cdf_min = dist.cdf(minimum)
    # cdf_max = dist.cdf(maximum)
    # mass = cdf_max - cdf_min
    # if mass <= 0:
    #     loc = minimum
    #     scale = maximum - minimum
    dist = beta(a, b, loc=loc, scale=scale)
    value = dist.ppf(s)
    return value, dist.pdf(value)

def _sample_from_pdf(s, param):
    minimum, maximum = param["min"], param["max"]
    if minimum == maximum:
        return minimum, 1.0
        
    match param.get("method", "uniform"):
        case "truncnorm":
            return _truncnorm_sample(
                s, param["a"], param["b"], param["loc"], param["scale"]
            )
        case "beta":
            return _beta_sample(
                s, param["a"], param["b"], param["loc"], param["scale"]
            )
        case _:
            return _uniform_sample(s, minimum, maximum)

def _affine_value(reference, values):
    if isinstance(reference, dict):
        return reference["factor"] * values[reference["parameter"]] + reference["offset"]
    return reference


def _sample_dependent(s, config, values):
    # Resolve affine fields for this sample using local values only.
    minimum = _affine_value(config.get("afin_min", config["min"]), values)
    maximum = _affine_value(config.get("afin_max", config["max"]), values)
    peak = _affine_value(config.get("afin_default", config["default"]), values)

    minimum, maximum = max(minimum, config["min"]), min(maximum, config["max"])

    # TODO: No hay control de errores si hay una mala configuración. En el caso anterior,
    # tampoco, pero aquí puede ser más fastidioso de entcontar.
    if minimum >= maximum: return minimum, 1.0
    if peak < minimum: peak = minimum
    elif peak > maximum: peak = maximum


    match config["method"]:
        case "beta":
            scale = maximum - minimum
            relative_peak = (peak - minimum) / scale
            a = 1 + relative_peak * config["lambda"]
            b = 1 + (1 - relative_peak) * config["lambda"]
            return _beta_sample(s, a, b, minimum, scale)
        case "truncnorm":
            std = (maximum - minimum) * config["std_factor"]
            a, b = (minimum - peak) / std, (maximum - peak) / std
            return _truncnorm_sample(s, a, b, peak, std)
        case _:
            return _uniform_sample(s, minimum, maximum)


def sample_params(params, n_samples, fordware=0, seed=42):
    sampler = qmc.Sobol(d=len(params), scramble=True, seed=seed)
    if fordware > 0:
        sampler.fast_forward(fordware)

    normal_params, dependent_params = [], []
    for i, (name, config) in enumerate(params.items()):
        selected = dependent_params if config.get("config_source") == "affine" else normal_params
        selected.append((i, name, config))

    for s in sampler.random(n_samples):
        values, densities = {}, {}
        for i, name, config in normal_params:
            value, density = _sample_from_pdf(s[i], config)
            values[name] = value
            densities[name] = density

        for i, name, config in dependent_params:
            value, density = _sample_dependent(s[i], config, values)
            values[name] = value
            densities[name] = density

        yield values, densities

        # # Mas elegante, pero menos eficiente
        # values, densities = zip(*map(_sample_from_pdf, s, params.values()))
        # yield dict(zip(params, values)), dict(zip(params, densities))



points, pdfs = map(list, zip(*sample_params(PARAMS, n_samples=1024, fordware=0, seed=42)))
pprint(f'Sampled {len(points)}')

In [ ]:
def plot_samples(configs, x_name, y_name):
    fig, axes = plt.subplots(2, 3, figsize=(12, 4), sharex=True, sharey=True, constrained_layout=True)

    for ax, (title, config) in zip(axes.T.flat, configs):
        points, pdfs = map(list, zip(*sample_params(config, n_samples=1024)))
        scatter = ax.scatter(
            [p[x_name] for p in points], [p[y_name] for p in points],
            c=[p[x_name] * p[y_name] for p in pdfs], cmap='viridis'
        )
        ax.set(xlabel=x_name, ylabel=y_name, title=title)
        fig.colorbar(scatter, ax=ax)

    plt.show()


configs = [
    ('Truncada, σ = 0.1 × rango', add_truncnorm_config(PARAMS, std_factor=0.1)),
    ('Truncada, σ = 0.3 × rango', add_truncnorm_config(PARAMS, std_factor=0.3)),
    ('Beta, λ = 1', add_beta_config(PARAMS, lambda_=1)),
    ('Beta, λ = 1.5', add_beta_config(PARAMS, lambda_=1.5)),
    ('Beta, λ = 3', add_beta_config(PARAMS, lambda_=3)),
    ('Uniform', PARAMS)
]
x_name, y_name = list(PARAMS)[:2]

plot_samples(configs, x_name, y_name)


### PDF sampling examples

Compare simple and PDF configurations, safe variants, and clamp bounds using
`plot_samples`, the same function as the original example.


In [ ]:
pdf_params = {
    'X': {'min': 10, 'max': 20, 'default': 14},
    'Y': {'min': 0, 'max': 1, 'default': 0.5},
}
pdf_configs = [
    ('Truncated normal, factor=0.1', add_truncnorm_config(pdf_params, std_factor=0.1, names=['X'])),
    ('Truncated normal PDF, scale=1', add_truncnorm_config_pdf(pdf_params, a=-4, b=6, loc=14, scale=1, names=['X'])),
    ('Truncated normal, factor=0.2', add_truncnorm_config(pdf_params, std_factor=0.2, names=['X'])),
    ('Truncated normal PDF, scale=2', add_truncnorm_config_pdf(pdf_params, a=-2, b=3, loc=14, scale=2, names=['X'])),
    ('Beta, lambda=5', add_beta_config(pdf_params, lambda_=5, names=['X'])),
    ('Beta PDF, a=3, b=4', add_beta_config_pdf(pdf_params, a=3, b=4, loc=10, scale=10, names=['X'])),
]

plot_samples(pdf_configs, 'X', 'Y')


In [ ]:
safe_params = {
    'V1': {'min': 0.2, 'max': 0.8, 'default': 0.5},
    'V2': {'min': [0.1, 0.2], 'max': [0.9, 0.8], 'default': [0.4, 0.6]},
}
safe_configs = [
    ('Truncated normal safe, factor=0.1', add_truncnorm_config_safe(safe_params, std_factor=0.1)),
    ('Truncated normal PDF safe, scale=0.2', add_truncnorm_config_pdf_safe(safe_params, a=-3, b=3, loc=0.5, scale=0.2)),
    ('Truncated normal safe, factor=0.3', add_truncnorm_config_safe(safe_params, std_factor=0.3)),
    ('Truncated normal PDF safe, scale=0.4', add_truncnorm_config_pdf_safe(safe_params, a=-3, b=3, loc=0.5, scale=0.4)),
    ('Beta safe, lambda=3', add_beta_config_safe(safe_params, lambda_=3)),
    ('Beta PDF safe, a=2, b=3', add_beta_config_pdf_safe(safe_params, a=2, b=3, loc=0, scale=1)),
]

plot_samples([(title, flatten_params(config)) for title, config in safe_configs], 'V2*00', 'V2*01')


In [ ]:
wide_params = {
    'V1': {'min': 0, 'max': 1, 'default': 0.5},
    'V2': {'min': 0, 'max': 1, 'default': 0.5},
}
clamp_params = {
    'V1': {'min': 0.25, 'max': 0.75, 'default': 0.5},
    'V2': {'min': 0.25, 'max': 0.75, 'default': 0.5},
}
clamp_configs = [
    ('Truncated normal, bounds [0, 1]', add_truncnorm_config_pdf(wide_params, a=-3, b=3, loc=0.5, scale=0.3)),
    ('Truncated normal, clamp [0.25, 0.75]', add_truncnorm_config_pdf(clamp_params, a=-3, b=3, loc=0.5, scale=0.3)),
    ('Beta (2, 3), bounds [0, 1]', add_beta_config_pdf(wide_params, a=2, b=3, loc=0, scale=1)),
    ('Beta (2, 3), clamp [0.25, 0.75]', add_beta_config_pdf(clamp_params, a=2, b=3, loc=0, scale=1)),
    ('Beta (3, 2), bounds [0, 1]', add_beta_config_pdf(wide_params, a=3, b=2, loc=0, scale=1)),
    ('Beta (3, 2), clamp [0.25, 0.75]', add_beta_config_pdf(clamp_params, a=3, b=2, loc=0, scale=1)),
]

plot_samples(clamp_configs, 'V1', 'V2')


### Dependent sampling examples


In [ ]:
dependent_params = {
    'X': {'min': 0.3, 'max': 0.7, 'default': 0.5},
    'Y': {'min': 0, 'max': 1, 'default': 0.5},
}
dependent_configs = [
    ('Beta, center=X', add_beta_config_dependent(dependent_params, lambda_=8, afin_default=affine('X'), names=['Y'])),
    ('Truncated normal, center=X', add_truncnorm_config_dependent(dependent_params, std_factor=0.1, afin_default=affine('X'), names=['Y'])),
    ('Uniform, center=X+0.2', add_uniform_config_dependent(dependent_params, afin_default=affine('X', offset=0.2), names=['Y'])),
    ('Beta, min=X-0.3', add_beta_config_dependent(dependent_params, lambda_=4, afin_min=affine('X', offset=-0.3), names=['Y'])),
    ('Truncated normal, max=0.5*X+0.6', add_truncnorm_config_dependent(dependent_params, std_factor=0.2, afin_max=affine('X', factor=0.5, offset=0.6), names=['Y'])),
    ('Uniform, bounds=X±0.2', add_uniform_config_dependent(dependent_params, afin_min=affine('X', offset=-0.2), afin_max=affine('X', offset=0.2), names=['Y'])),
]

plot_samples(dependent_configs, 'X', 'Y')


## Weights


Si se usan distribuciones de probabilidad al rededor de la media para el muestreo, así garantizamos que el modelo aprenda bien esa zona. No obstante, podemos aumentar la ponderación de las zonas poco probables para que apesar de tener pocas muestras, tengan un impacto en el entrenamiento del modelo.

**Pdf usada en el muestreo**

$$
Q(\mathbf{X}) = \prod_{i=1}^{n} q_i(x_i)
$$

**Si suponemos que la distribución de probabilidad de cada parámetro es uniforme y que las variables son independientes**, la distribucion real de la probabilidad puede calcularse como:

$$
\begin{align}
U(\mathbf{X}) = \prod_{i=1}^{n} u_i(x_i) \\
u_i(x_i) = cte \quad \forall x_i \\
U(\mathbf{X}) = cte \quad \forall \mathbf{X}
\end{align}
$$ 

**Las ténicas de simulación de Monte Carlo calculan un peso para cada muestra**. Este peso es el cociente entre la distribución real de probabilidad y la distribución usada en el muestreo:

$$
W(\mathbf{X}) = \frac{U(\mathbf{X})}{Q(\mathbf{X})}
$$

En DL, como el la distribución de probablidad es constante:

$$
W(\mathbf{X}) = \frac{U(\mathbf{X})}{Q(\mathbf{X})}
\overset{\mathrm{optimization}}{\equiv}
\frac{1}{Q(\mathbf{X})}
$$

En nuestro modelo DL, (i) no sabemos cual es la distribución real (puede que esta no este tan alejada de la de muestreo), (ii) puede que dividir por la distribución de muestreo de demasiado peso a valores extremos. Lo podemos solucionar considernado que la distribución real es una distribución que se obtiene de interpolar la distribución de muestreo con una distribución uniforme. 


$$
P(\mathbf{X}) = (1 - \alpha) Q(\mathbf{X}) + \alpha U(\mathbf{X})
$$

De esta forma el peso quedaría de la siguiente forma:

$$
W(\mathbf{X}) = (1-\alpha) + \alpha \frac{U(\mathbf{X})}{Q(\mathbf{X})} 
\overset{\mathrm{optimization}}{\equiv} 
(1-\alpha) + \alpha \frac{1}{Q(\mathbf{X})}
$$




### Dos parámetros, multiples distribuciones

In [ ]:
PARAMS = {
    'V1': {'min': 0, 'max': 1, 'default': 0.25},
    'V2': {'min': 2, 'max': 10, 'default': 7.5},
}

sample_configs = [
    ('Truncada, σ=0.2', add_truncnorm_config(PARAMS, 0.2)),
    ('Truncada, σ=0.35', add_truncnorm_config(PARAMS, 0.35)),
    ('Beta, λ=3', add_beta_config(PARAMS, 3)),
    ('Uniforme', PARAMS),
]

sample_set = [(name, list(sample_params(config, 512))) for name, config in sample_configs]

In [ ]:
def sample_weight(pdfs, alpha=0.5):
    return (1 - alpha) + alpha / np.prod(list(pdfs.values()))

def weight_norm(weights, lognorm):
    if not lognorm:
        return None
    lo, hi = min(weights), max(weights)
    if lo == hi:
        lo, hi = lo / 1.1, hi * 1.1
    return LogNorm(vmin=lo, vmax=hi)

In [ ]:

@interact(alpha=FloatSlider(value=0.5, min=0, max=1, step=0.05,
                             description='α', continuous_update=False),
          lognorm=ToggleButton(value=False, description='LogNorm'))
def plot_weights(alpha, lognorm):
    weights = [[sample_weight(pdf, alpha) for _, pdf in samples]
               for _, samples in sample_set]
    
    all_weights = np.concatenate(weights)

    pprint(f'Weight shape: {all_weights.shape}')

    fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True,
                             sharey=True, constrained_layout=True)
    for ax, (name, samples), color in zip(axes.flat, sample_set, weights):
        points = [point for point, _ in samples]
        plot = ax.scatter([p['V1'] for p in points],
                          [p['V2'] for p in points], c=color,
                          norm=weight_norm(color, lognorm),
                          cmap='viridis', s=10)
        ax.set(xlabel='V1', ylabel='V2', title=name)
        fig.colorbar(plot, ax=ax, label='Peso W')
    plt.show()

### Beta, 10 parámetros, 1024 muestras y proyección PCA

In [ ]:
rng = np.random.default_rng(42)
mins = rng.uniform(0, 7, 10)
maxs = mins + rng.uniform(1, 3, 10)
relative_peaks = rng.uniform(0.2, 0.8, 10)

PARAMS_BETA10 = {
    f'V{i + 1}': {'min': lo, 'max': hi,
                  'default': lo + relative_peak * (hi - lo)}
    for i, (lo, hi, relative_peak) in enumerate(
        zip(mins, maxs, relative_peaks))
}

beta10_samples = list(sample_params(add_beta_config(PARAMS_BETA10, 3),
                                    n_samples=1024))
beta10_values = np.array([[values[name] for name in PARAMS_BETA10]
                          for values, _ in beta10_samples])
beta10_pca = PCA(n_components=2)
beta10_xy = beta10_pca.fit_transform(beta10_values)

@interact(alpha=FloatSlider(value=0.5, min=0, max=1, step=0.05,
                             description='α', continuous_update=False),
          lognorm=ToggleButton(value=False, description='LogNorm'))
def plot_beta10_pca(alpha, lognorm):
    weights = [sample_weight(pdfs, alpha) for _, pdfs in beta10_samples]
    fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
    plot = ax.scatter(beta10_xy[:, 0], beta10_xy[:, 1], c=weights,
                      norm=weight_norm(weights, lognorm),
                      cmap='viridis', s=12)
    ax.set_title('Beta: 10 parámetros, 1024 muestras')
    fig.colorbar(plot, ax=ax, label='Peso W')
    plt.show()